In [ ]:
import os
import json
import urllib.request
import urllib.error
import openai
from dotenv import load_dotenv
load_dotenv()  # .env 파일 자동 로드

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def _get(path: str):
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (compatible; AI-Client/1.0)",
    }
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = resp.read().decode("utf-8")
        return json.loads(data)
    except urllib.error.HTTPError as e:
        return {"error": f"HTTP {e.code}", "url": url}
    except urllib.error.URLError as e:
        return {"error": f"URL error: {e.reason}", "url": url}


def get_popular_movies():
    return json.dumps(_get("/movies"), ensure_ascii=False)


def get_movie_details(id):
    return json.dumps(_get(f"/movies/{id}"), ensure_ascii=False)


def get_movie_credits(id):
    return json.dumps(_get(f"/movies/{id}/credits"), ensure_ascii=False)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
}

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [ ]:
SYSTEM_PROMPT = """
너는 Movie Expert Agent이다.
사용자의 질문을 이해하고 아래 함수 중 하나를 선택해 호출하라.
필요할 때만 함수를 호출하고, 함수명과 인자를 정확히 지정하라.

사용 가능한 함수:
- get_popular_movies(): /movies에서 인기 영화 목록을 가져온다.
- get_movie_details(id): /movies/:id에서 영화 정보를 가져온다.
- get_movie_credits(id): /movies/:id/credits에서 출연진 및 제작진을 가져온다.
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 영화 목록을 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 영화 상세 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화 ID로 출연진 및 제작진 정보를 가져온다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 ID",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

messages = [{"role": "system", "content": SYSTEM_PROMPT}]


In [ ]:
def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": message})
    print(f"AI: {message}")


In [ ]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()
'''
User: 나는 SF 영화를 좋아해
AI: 좋아하는 SF 영화에 대한 정보를 제공하기 위해 인기 SF 영화를 찾겠습니다.

get_popular_movies() 호출하겠습니다.
User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 인셉션과 인터스텔라를 이미 보셨군요! 그러면, 비슷한 SF 영화 추천을 위해 인기 영화 목록에서 더 많은 정보를 찾아보겠습니다.

get_popular_movies() 호출하겠습니다.
User: 오늘 밤에 뭐 볼지 추천해 줄래?
AI: 인기 영화 목록에서 추천할 만한 SF 영화를 찾아보겠습니다.

get_popular_movies() 호출하겠습니다.
User: 내가 좋아하는 장르랑 이미 본 영화가 뭐라고 했지?
AI: 사용자가 좋아하는 장르는 SF 영화이며, 이미 본 영화는 "인셉션"과 "인터스텔라"라고 하셨습니다.
'''